In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import MBartTokenizer, MBartForConditionalGeneration, Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch
from evaluate import load
import random
from tqdm import tqdm
from collections import Counter
from difflib import SequenceMatcher

# Load the dataset
data_path = "/kaggle/input/erupd-nmt/ERUPD_NMT.csv"
df = pd.read_csv(data_path, encoding='latin')  # Load with Latin1 encoding

# Split the data
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=42)
test_df_size = 200 / len(temp_df)
valid_df, test_df = train_test_split(temp_df, test_size=test_df_size, random_state=42)

In [3]:
df.head()

,English,Roman Urdu
0,"In the heart of the bustling city, lived a you...",Shehar ki dil mein rehti thi ek nojawan aurat ...
1,She had always dreamed of exploring the world ...,Usne hamesha khwab dekha tha ke apne sheher ke...
2,"One day, as she perused a travel magazine, Ais...","Ek din, jab usne ek safar nama parha, Aisha ne..."
3,The vivid descriptions of lush landscapes and ...,Yehan ke hari bhari manazir aur gaon walon ki ...
4,"Determined to turn her dreams into reality, Ai...",Apne khwabon ko haqeeqat mein tabdeel karne ka...


In [4]:
# Prepare data
train_texts = train_df['English'].tolist()
train_labels = [f"<roman_urdu> {label}" for label in train_df['Roman Urdu'].tolist()]
valid_texts = valid_df['English'].tolist()
valid_labels = [f"<roman_urdu> {label}" for label in valid_df['Roman Urdu'].tolist()]
test_texts = test_df['English'].tolist()
test_labels = [f"<roman_urdu> {label}" for label in test_df['Roman Urdu'].tolist()]



In [ ]:
from transformers import MBartTokenizer, MBartForConditionalGeneration

# Load mBART model and tokenizer
model_name = "facebook/mbart-large-50"
tokenizer = MBartTokenizer.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)


# Add custom token for Roman Urdu
tokenizer.add_special_tokens({'additional_special_tokens': ['<roman_urdu>']})
model.resize_token_embeddings(len(tokenizer))

# Set language tokens
tokenizer.src_lang = "en_XX" 
tokenizer.tgt_lang = "<roman_urdu>"  

# Tokenize function
def tokenize_data(texts, labels, tokenizer, max_length=128):
    texts = [str(text) for text in texts]
    labels = [str(label) for label in labels]
    inputs = tokenizer(texts, max_length=max_length, padding=True, truncation=True, return_tensors="pt")
    targets = tokenizer(labels, max_length=max_length, padding=True, truncation=True, return_tensors="pt")
    return inputs, targets

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizer'.
/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, inputs, targets):
        self.inputs = inputs
        self.targets = targets

    def __len__(self):
        return len(self.inputs["input_ids"])

    def __getitem__(self, idx):
        input_ids = self.inputs["input_ids"][idx]
        attention_mask = self.inputs["attention_mask"][idx]
        target_ids = self.targets["input_ids"][idx]
        return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": target_ids}


train_inputs, train_targets = tokenize_data(train_texts, train_labels, tokenizer)
valid_inputs, valid_targets = tokenize_data(valid_texts, valid_labels, tokenizer)


train_dataset = TranslationDataset(train_inputs, train_targets)
valid_dataset = TranslationDataset(valid_inputs, valid_targets)


training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    learning_rate=5e-5,
    report_to="none",
    fp16=True
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset
)


trainer.train()



/opt/conda/lib/python3.10/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss
1,0.301300,0.289025
2,0.208100,0.248423
3,0.127000,0.253458


/opt/conda/lib/python3.10/site-packages/transformers/modeling_utils.py:2618: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=90174, training_loss=0.26141820099972113, metrics={'train_runtime': 33706.4181, 'train_samples_per_second': 5.351, 'train_steps_per_second': 2.675, 'total_flos': 4.885468992346522e+16, 'train_loss': 0.26141820099972113, 'epoch': 3.0})

In [ ]:
# BLEU metric loading
bleu = load("bleu")

# METEOR Score
def custom_meteor_score(reference, hypothesis):
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()
    ref_counts = Counter(ref_tokens)
    hyp_counts = Counter(hyp_tokens)
    matches = sum(min(ref_counts[word], hyp_counts[word]) for word in hyp_counts)
    precision = matches / len(hyp_tokens) if hyp_tokens else 0
    recall = matches / len(ref_tokens) if ref_tokens else 0
    f_score = (10 * precision * recall) / (9 * precision + recall) if precision + recall > 0 else 0
    matcher = SequenceMatcher(None, ref_tokens, hyp_tokens)
    match_blocks = matcher.get_matching_blocks()
    fragmentation = sum(1 for i in range(len(match_blocks) - 1) if match_blocks[i].size > 0)
    penalty = 0.5 * (fragmentation / len(hyp_tokens)) if hyp_tokens else 1
    return f_score * (1 - penalty)

# Evaluate model
def evaluate_model(test_texts, test_labels, model, tokenizer, sample_size=200):
    sample_size = min(sample_size, len(test_texts))
    sample_indices = random.sample(range(len(test_texts)), sample_size)
    sample_test_texts = [test_texts[i] for i in sample_indices]
    sample_test_labels = [test_labels[i] for i in sample_indices]

    sample_inputs, _ = tokenize_data(sample_test_texts, sample_test_labels, tokenizer)
    sample_inputs = {k: v.to(model.device) for k, v in sample_inputs.items()}

    predictions = []
    for input_id in tqdm(sample_inputs['input_ids'], desc="Processing"):
        output = model.generate(input_id.unsqueeze(0))
        predictions.append(output[0])

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = sample_test_labels

    bleu_score = bleu.compute(predictions=decoded_preds, references=[[label] for label in decoded_labels])['bleu']
    meteor_scores = [custom_meteor_score(ref, hyp) for ref, hyp in zip(decoded_labels, decoded_preds)]
    avg_meteor_score = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0

    return {"bleu": bleu_score, "meteor": avg_meteor_score}

# Evaluate
metrics = evaluate_model(test_texts, test_labels, model, tokenizer)
print(f"BLEU Score: {metrics['bleu']}")
print(f"Custom METEOR Score: {metrics['meteor']}")

Processing: 100%|██████████| 200/200 [03:23<00:00,  1.02s/it]


BLEU Score: 0.38770207386288646
Custom METEOR Score: 0.5311081695442068


In [8]:
def translate_text(text, model, tokenizer):
    text = f"{tokenizer.src_lang} {text}"
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(model.device)
    output = model.generate(inputs['input_ids'])
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
# Function to translate text(s)
def translate_text(texts, model, tokenizer):
    """
    Translate a single text or a list of texts.
    
    Args:
        texts (str or list): Input text(s) to be translated.
        model: The trained translation model.
        tokenizer: Tokenizer for the model.
        
    Returns:
        list: List of translated texts.
    """
    if isinstance(texts, str):
        texts = [texts] 
    
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    outputs = model.generate(**inputs)
    translated_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return translated_texts

# Example translations
example_texts = ["Hello, how are you?", "What is your name?", "Sachin Tendulkar is one of the greatest Cricket player","Everybody loves to spend time time with their family","it is better for students to follow their dreams","Work hard for something and you will reach success","I live in cheltenham","Arsenal vs Spurs is a proper Football match","The goal of my life is to bring justice to the people and bring about a positive change","Who cares what people say about you"]
translated_texts = translate_text(example_texts, model, tokenizer)

# Display translations
for i, text in enumerate(example_texts):
    print(f"Original: {text}")
    print(f"Translated: {translated_texts[i]}\n")


Original: Hello, how are you?
Translated: Hello, kaise ho?

Original: What is your name?
Translated: Aap ka naam kya hai?

Original: Sachin Tendulkar is one of the greatest Cricket player
Translated: Sachin Tendulkar behtareen cricket khiladi

Original: Everybody loves to spend time time with their family
Translated: Everybody loves to spend time time with their family

Original: it is better for students to follow their dreams
Translated: behtar hai talba ke liye apne khwabon ko follow karen

Original: Work hard for something and you will reach success
Translated: mehnat kar ke aap kamyabi tak pohnch jayenge

Original: I live in cheltenham
Translated: Main cheltenham mein rehta hoon

Original: Arsenal vs Spurs is a proper Football match
Translated: Arsenal vs Spurs sahi Football match hai

Original: The goal of my life is to bring justice to the people and bring about a positive change
Translated: Meri zindagi ka maqsad logon ko insaaf dena aur ek musbat tabdeeli laane ka hai

Origina

In [ ]:
trainer.save_model("./checkpoint-3-epochs") 
